# 05 · Fase 1: efecto propio y efectos dentro de la categoría

Estima, para una categoría objetivo $j$, el modelo PPML de la **Fase 1** de `metodologia.tex`: efecto de la promoción del propio SKU (día actual y dos bloques de rezagos) y de la promoción de los otros SKUs de su clase y del resto de su categoría.

| | |
|---|---|
| **Entrada** | `panel_t44_sku_dia.parquet`, `exposiciones_t44_sku_dia.parquet`, `calendario_t44.csv` (en `../data/procesados/`) |
| **Salida** | `resultados/tablas/fase1_<CATEGORIA>.csv` y `resultados/tablas/fase1_consolidada.csv` |
| **Especificación** | PPML, variable dependiente `unit_sales_ppml`; efectos fijos SKU, día de la semana y mes; errores agrupados por SKU y semana |

**Cómo usarlo.** Empiecen con `categorias_objetivo = ["BEVERAGES"]`, revisen los controles de la sección 3 y recién entonces agreguen las demás categorías a la lista. No se incluye `transactions` ni el petróleo.

In [7]:
# ------------------------------------------------------------------------------------------------------------
# Este notebook estima el modelo PPML de la fase 1 (efecto propio y dentro de la categoría) por categoría.
# ------------------------------------------------------------------------------------------------------------

# Librerías
# ------------------------------------------------------------------------------------------------------------
import os
import re

import numpy as np
import pandas as pd
import pyfixest as pf

## 1. Parámetros

In [8]:
# Categorías a estimar (empezar solo con BEVERAGES; luego agregar las demás de la lista de prioridad)
# ------------------------------------------------------------------------------------------------------------
categorias_objetivo = ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER"]
# Lista completa por prioridad (INSTRUCCIONES.md, sección 6):
# ["BEVERAGES", "GROCERY I", "CLEANING", "DAIRY", "BREAD/BAKERY", "LIQUOR,WINE,BEER",
#  "MEATS", "POULTRY", "DELI", "PRODUCE"]

# Categorías con programa semanal fijo (D-26): se excluyen sus días de programa
categorias_con_programa = ["MEATS", "POULTRY", "SEAFOOD", "EGGS", "DELI", "PRODUCE"]

# Bloques de promoción (D-14): día actual y dos bloques de rezagos
sufijos = ["", "_l1_3", "_l4_6"]

# Indicadores de calendario (h_t); 'terremoto' queda fuera porque su ventana se excluye de la muestra
indicadores_calendario = [
    "dia_libre", "adicional", "dia_laboral_recuperado", "evento_comercial", "evento_otro", "dia_pago", "post_pago",
]

# Efectos fijos y agrupamiento de los errores estándar (D-15, D-16)
efectos_fijos = ["item_nbr", "dow", "mes"]
agrupar_por = ["item_nbr", "semana"]

# Días máximos de estimación: 590 menos 46 de la ventana del terremoto
dias_maximos = 544

# Rutas
# ------------------------------------------------------------------------------------------------------------
ruta_procesados = os.path.join("..", "data", "procesados")
ruta_tablas = os.path.join("resultados", "tablas")
os.makedirs(ruta_tablas, exist_ok=True)
archivo_panel = os.path.join(ruta_procesados, "panel_t44_sku_dia.parquet")
archivo_exposiciones = os.path.join(ruta_procesados, "exposiciones_t44_sku_dia.parquet")
archivo_calendario = os.path.join(ruta_procesados, "calendario_t44.csv")

## 2. Base de estimación de una categoría

Sigue los pasos de la última celda de `03_exposiciones` (sin las exposiciones a otras categorías, que entran en la fase 2). Cada unión verifica que no se dupliquen ni se pierdan filas, y cada exclusión se cuenta.

In [9]:
def construir_base(categoria: str) -> tuple:
    """
    Esta función arma la base de estimación de la fase 1 para una categoría objetivo.

    :param categoria:   el nombre de la categoría objetivo (p. ej., 'BEVERAGES').
    :return:            una tupla (base, exclusiones): la base SKU-día lista para estimar y un diccionario con el
                        número de filas en cada paso.
    """

    exclusiones = {}

    # 1. Filas del panel de la categoría
    panel = pd.read_parquet(
        archivo_panel,
        columns=["date", "item_nbr", "family", "unit_sales_ppml", "onpromotion"],
        filters=[("family", "==", categoria)],
    )
    panel["family"] = panel["family"].astype(str)
    assert len(panel) > 0, f"no hay filas para la categoría {categoria}"
    assert not panel.duplicated(["item_nbr", "date"]).any(), "el panel tiene más de una fila por SKU-día"
    exclusiones["panel de la categoría"] = len(panel)

    # 2. Unir las exposiciones SKU-día (uno a uno por SKU y fecha)
    columnas_expo = ["item_nbr", "date", "dia_programa", "promo_clase_otros", "promo_cat_otras_clases"] + [
        f"{v}{s}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for s in sufijos[1:]
    ]
    expo = pd.read_parquet(archivo_exposiciones, columns=columnas_expo)
    n0 = len(panel)
    base = panel.merge(expo, on=["item_nbr", "date"], how="left", validate="one_to_one")
    assert len(base) == n0, "la unión con exposiciones cambió el número de filas"
    assert base["dia_programa"].notna().all(), "hay SKU-días del panel sin exposiciones"

    # 3. Unir el calendario (muchos a uno por fecha); las fechas se leen con parse_dates
    calendario = pd.read_csv(archivo_calendario, parse_dates=["date"])
    base = base.merge(
        calendario[["date", "dow", "mes", "semana", "ventana_terremoto"] + indicadores_calendario],
        on="date", how="left", validate="many_to_one",
    )
    assert len(base) == n0, "la unión con el calendario cambió el número de filas"
    assert base["dow"].notna().all(), "hay fechas del panel sin calendario"
    assert not base.duplicated(["item_nbr", "date"]).any(), "la base tiene más de una fila por SKU-día"

    # 4. Excluir la ventana del terremoto
    base = base.loc[~base["ventana_terremoto"]]
    exclusiones["tras excluir ventana del terremoto"] = len(base)

    # 5. Si la categoría tiene programa semanal, excluir sus días de programa
    if categoria in categorias_con_programa:
        base = base.loc[~base["dia_programa"]]
    exclusiones["tras excluir días de programa"] = len(base)

    # 6. Excluir filas con nulos en las variables del modelo
    base["onpromotion"] = base["onpromotion"].astype(float)
    variables = variables_modelo()
    base = base.dropna(subset=variables + ["unit_sales_ppml"])
    exclusiones["tras excluir nulos (base final)"] = len(base)

    assert not base.duplicated(["item_nbr", "date"]).any(), "la base final tiene más de una fila por SKU-día"
    return base.reset_index(drop=True), exclusiones


def variables_modelo() -> list:
    """
    Esta función devuelve la lista de regresores de la fase 1 (promoción propia, de la clase y del resto de la
    categoría, cada una con sus bloques de rezagos).

    :return:    la lista de nombres de columna.
    """

    return [f"{v}{s}" for v in ["onpromotion", "promo_clase_otros", "promo_cat_otras_clases"] for s in sufijos]

## 3. Estimación y tabla de resultados

PPML con `pyfixest.fepois`. Se descartan los indicadores de calendario sin variación en la muestra (p. ej., `terremoto` ya no existe, y un feriado puede caer solo en días excluidos). El efecto neto de la promoción propia es $e^{\beta_0+\beta_{\ell_{1-3}}+\beta_{\ell_{4-6}}}-1$, con su error estándar por el método delta.

In [10]:
def estimar_fase1(base: pd.DataFrame, categoria: str) -> pd.DataFrame:
    """
    Esta función estima el PPML de la fase 1 y devuelve la tabla de resultados.

    :param base:        la base de estimación de la categoría (salida de construir_base).
    :param categoria:   el nombre de la categoría objetivo.
    :return:            un DataFrame con una fila por coeficiente de interés (y una fila de efecto neto propio) con
                        coeficiente, error estándar, valor p, efecto en porcentaje y tamaño de la muestra.
    """

    # Conservar solo los indicadores de calendario con variación
    calendario_activo = [c for c in indicadores_calendario if base[c].nunique() > 1]
    descartados = sorted(set(indicadores_calendario) - set(calendario_activo))
    if descartados:
        print(f"  Indicadores sin variación (descartados): {descartados}")
    base = base.assign(**{c: base[c].astype(float) for c in calendario_activo})

    # Estimar
    regresores = variables_modelo() + calendario_activo
    formula = f"unit_sales_ppml ~ {' + '.join(regresores)} | {' + '.join(efectos_fijos)}"
    modelo = pf.fepois(formula, data=base, vcov={"CRV1": "+".join(agrupar_por)})

    # Reunir los coeficientes de interés
    tabla = modelo.tidy().reset_index().rename(columns={"Coefficient": "variable"})
    tabla = tabla.rename(columns={"Estimate": "coef", "Std. Error": "se", "Pr(>|t|)": "p_valor"})
    tabla = tabla[["variable", "coef", "se", "p_valor"]]

    # Efecto neto propio con el método delta: combinación lineal de tres coeficientes
    nombres = list(modelo._coefnames)
    propios = [f"onpromotion{s}" for s in sufijos]
    w = np.array([1.0 if n in propios else 0.0 for n in nombres])
    vcov = np.asarray(modelo._vcov)
    coef_neto = float(w @ modelo.coef().reindex(nombres).to_numpy())
    se_neto = float(np.sqrt(w @ vcov @ w))
    from scipy import stats
    p_neto = float(2 * (1 - stats.norm.cdf(abs(coef_neto / se_neto))))
    tabla = pd.concat(
        [tabla, pd.DataFrame([{"variable": "efecto_neto_propio", "coef": coef_neto, "se": se_neto, "p_valor": p_neto}])],
        ignore_index=True,
    )

    # Efecto en porcentaje, 100 (exp(beta) - 1), e intervalo de confianza al 95%
    tabla["efecto_pct"] = 100 * (np.exp(tabla["coef"]) - 1)
    tabla["ic95_inf_pct"] = 100 * (np.exp(tabla["coef"] - 1.96 * tabla["se"]) - 1)
    tabla["ic95_sup_pct"] = 100 * (np.exp(tabla["coef"] + 1.96 * tabla["se"]) - 1)

    # Tamaño de la muestra
    tabla.insert(0, "categoria", categoria)
    tabla["n_obs"] = len(base)
    tabla["n_skus"] = base["item_nbr"].nunique()
    tabla["n_dias"] = base["date"].nunique()

    # Solo se reportan las variables de promoción (los indicadores de calendario son controles)
    interes = variables_modelo() + ["efecto_neto_propio"]
    return tabla.loc[tabla["variable"].isin(interes)].reset_index(drop=True)

In [11]:
# Estimar cada categoría y guardar su tabla
resultados = {}
for categoria in categorias_objetivo:
    print(f"\n=== {categoria} ===")
    base, exclusiones = construir_base(categoria)

    # Controles de la sección 10.1 de INSTRUCCIONES.md
    for paso, n in exclusiones.items():
        print(f"  {paso:<40} {n:>10,}")
    dias_usados = base["date"].nunique()
    assert dias_usados <= dias_maximos, f"se usan {dias_usados} días y el máximo es {dias_maximos}: falta una exclusión"
    print(f"  SKUs: {base['item_nbr'].nunique():,} | días usados: {dias_usados} (máximo {dias_maximos})")

    tabla = estimar_fase1(base, categoria)
    resultados[categoria] = tabla
    nombre_archivo = re.sub(r"[^A-Za-z0-9]+", "_", categoria).strip("_")
    tabla.to_csv(os.path.join(ruta_tablas, f"fase1_{nombre_archivo}.csv"), index=False)

    # El efecto propio contemporáneo debería ser positivo; si no, revisar antes de interpretar
    propio = tabla.loc[tabla["variable"] == "onpromotion"].iloc[0]
    print(tabla[["variable", "coef", "se", "p_valor", "efecto_pct"]].round(4).to_string(index=False))
    if propio["coef"] <= 0:
        print("  >>> ATENCIÓN: el efecto propio del día es <= 0; revisar la construcción de la variable antes de interpretar")
    if (tabla["coef"].abs() > 5).any():
        print("  >>> ATENCIÓN: hay coeficientes con |valor| > 5 (posible colinealidad o separación)")


=== BEVERAGES ===
  panel de la categoría                       287,407
  tras excluir ventana del terremoto          266,744
  tras excluir días de programa               266,744
  tras excluir nulos (base final)             258,915
  SKUs: 576 | días usados: 532 (máximo 544)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1 singleton fixed effect(s) dropped from the model.
  warnings.warn(


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.4688 0.0580   0.0000     59.8155
           onpromotion_l1_3  0.2384 0.0618   0.0001     26.9232
           onpromotion_l4_6 -0.1045 0.0627   0.0955     -9.9227
          promo_clase_otros -0.3055 0.1143   0.0075    -26.3266
     promo_clase_otros_l1_3 -0.0197 0.1204   0.8701     -1.9488
     promo_clase_otros_l4_6  0.0786 0.1014   0.4384      8.1747
     promo_cat_otras_clases -0.6335 0.3613   0.0796    -46.9260
promo_cat_otras_clases_l1_3  0.2720 0.4135   0.5106     31.2614
promo_cat_otras_clases_l4_6  0.4063 0.2976   0.1722     50.1306
         efecto_neto_propio  0.6028 0.0781   0.0000     82.7154

=== GROCERY I ===
  panel de la categoría                       620,919
  tras excluir ventana del terremoto          574,218
  tras excluir días de programa               574,218
  tras excluir nulos (base final)             556,533
  SKUs: 1,153 | días usados: 532 (máximo 544)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2 singleton fixed effect(s) dropped from the model.
  warnings.warn(


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.4727 0.0507   0.0000     60.4291
           onpromotion_l1_3  0.2597 0.0428   0.0000     29.6526
           onpromotion_l4_6 -0.0109 0.0431   0.8005     -1.0837
          promo_clase_otros -0.2255 0.0717   0.0017    -20.1904
     promo_clase_otros_l1_3 -0.3140 0.0735   0.0000    -26.9496
     promo_clase_otros_l4_6  0.1692 0.0750   0.0240     18.4306
     promo_cat_otras_clases -0.7388 0.3946   0.0612    -52.2303
promo_cat_otras_clases_l1_3 -0.3999 0.6613   0.5453    -32.9644
promo_cat_otras_clases_l4_6  1.2656 0.6077   0.0373    254.5307
         efecto_neto_propio  0.7215 0.0446   0.0000    105.7464

=== CLEANING ===
  panel de la categoría                       220,098
  tras excluir ventana del terremoto          203,350
  tras excluir días de programa               203,350
  tras excluir nulos (base final)             197,454
  SKUs: 396 | días usados: 532 (máximo 544)


c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.6363 0.0742   0.0000     88.9383
           onpromotion_l1_3  0.2331 0.1028   0.0234     26.2466
           onpromotion_l4_6 -0.0243 0.0692   0.7260     -2.3976
          promo_clase_otros -0.3105 0.1354   0.0218    -26.6889
     promo_clase_otros_l1_3 -0.4868 0.1979   0.0139    -38.5432
     promo_clase_otros_l4_6  0.1126 0.1488   0.4492     11.9200
     promo_cat_otras_clases -0.4895 0.3676   0.1829    -38.7086
promo_cat_otras_clases_l1_3  0.3007 0.4111   0.4644     35.0867
promo_cat_otras_clases_l4_6  0.7745 0.4284   0.0706    116.9493
         efecto_neto_propio  0.8450 0.0678   0.0000    132.8094

=== DAIRY ===
  panel de la categoría                       122,674
  tras excluir ventana del terremoto          113,450
  tras excluir días de programa               113,450
  tras excluir nulos (base final)             109,779
  SKUs: 226 | días usados: 532 (máximo 544)
                   va

c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


                   variable    coef     se  p_valor  efecto_pct
                onpromotion  0.3747 0.0584   0.0000     45.4517
           onpromotion_l1_3  0.1445 0.0661   0.0287     15.5460
           onpromotion_l4_6  0.0985 0.0749   0.1885     10.3544
          promo_clase_otros -0.0753 0.0893   0.3993     -7.2543
     promo_clase_otros_l1_3 -0.0494 0.0844   0.5582     -4.8207
     promo_clase_otros_l4_6 -0.1399 0.0991   0.1579    -13.0564
     promo_cat_otras_clases -0.6642 0.2305   0.0040    -48.5324
promo_cat_otras_clases_l1_3  0.4088 0.3625   0.2595     50.5014
promo_cat_otras_clases_l4_6 -0.1755 0.2529   0.4877    -16.0945
         efecto_neto_propio  0.6177 0.0991   0.0000     85.4658

=== LIQUOR,WINE,BEER ===
  panel de la categoría                        27,102
  tras excluir ventana del terremoto           25,148
  tras excluir días de programa                25,148
  tras excluir nulos (base final)              21,466
  SKUs: 49 | días usados: 532 (máximo 544)
           

c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))
c:\Users\Laury\conda\envs\Tesis_Promociones\Lib\site-packages\pyfixest\estimation\models\_result_accessor_mixin.py:85: RuntimeWarning: invalid value encountered in sqrt
  self._se = np.sqrt(np.diagonal(self._vcov))


## 4. Comparación con los descriptivos y tabla consolidada

La dirección del efecto propio debe parecerse a la razón de medias de la tabla 4 de `04_descriptivos` (la magnitud puede diferir, porque el modelo controla SKU y calendario).

In [12]:
# Comparar con la tabla 4 de los descriptivos (razón de medias con / sin promoción)
archivo_t4 = os.path.join(ruta_tablas, "tabla_04_ventas_con_sin_promocion.csv")
if os.path.exists(archivo_t4):
    t4 = pd.read_csv(archivo_t4, index_col="family")
    for categoria, tabla in resultados.items():
        if categoria in t4.index:
            efecto = tabla.loc[tabla["variable"] == "onpromotion", "efecto_pct"].iloc[0]
            print(f"{categoria}: modelo {efecto:+.1f}% | descriptivo (razón de medias) {t4.loc[categoria, 'razon_medias']:.2f}")
else:
    print("No se encontró la tabla 4; ejecutar 04_descriptivos para comparar.")

# Consolidar todas las tablas por categoría que existan en la carpeta (así no se pierde lo estimado antes)
archivos = sorted(f for f in os.listdir(ruta_tablas) if f.startswith("fase1_") and f != "fase1_consolidada.csv")
consolidada = pd.concat([pd.read_csv(os.path.join(ruta_tablas, f)) for f in archivos], ignore_index=True)
consolidada.to_csv(os.path.join(ruta_tablas, "fase1_consolidada.csv"), index=False)
print(f"\nConsolidada: {consolidada['categoria'].nunique()} categorías -> {os.path.join(ruta_tablas, 'fase1_consolidada.csv')}")

print(f"\n>>>> Proceso finalizado <<<<")

BEVERAGES: modelo +59.8% | descriptivo (razón de medias) 2.35
GROCERY I: modelo +60.4% | descriptivo (razón de medias) 2.36
CLEANING: modelo +88.9% | descriptivo (razón de medias) 2.36
DAIRY: modelo +41.2% | descriptivo (razón de medias) 1.28
BREAD/BAKERY: modelo +45.5% | descriptivo (razón de medias) 1.91
LIQUOR,WINE,BEER: modelo +75.0% | descriptivo (razón de medias) 2.20

Consolidada: 6 categorías -> resultados\tablas\fase1_consolidada.csv

>>>> Proceso finalizado <<<<
